Installing dependencies such as Ollama for LLM management and pyngrok to expose the API.

In [ ]:
# Install system dependencies + Ollama + pyngrok

!sudo apt-get update -qq
!sudo apt-get install -y -qq pciutils zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Verify installation
!which ollama
!ollama --version

# Install Python dependency
!pip install -q pyngrok requests

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
/usr/local/bin/ollama
ollama version is 0.33.2


Setting up code thread

In [ ]:
import os
import time
import shutil
import subprocess
import threading
import requests

OLLAMA_BIN = shutil.which("ollama") or "/usr/local/bin/ollama"
OLLAMA_API = "http://127.0.0.1:11434"

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_ORIGINS"] = "*"
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

print("Ollama binary:", OLLAMA_BIN)

ollama_process = subprocess.Popen(
    [OLLAMA_BIN, "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Drain the subprocess's stdout continuously in a background thread.
# Without this, the OS pipe buffer eventually fills up (Ollama keeps logging
# while the notebook runs) and the "serve" process blocks/hangs once it's full,
# which shows up later as mysterious timeouts.
def _drain_output(proc):
    for line in proc.stdout:
        pass  # discard; change to print(line, end="") if you want live logs

threading.Thread(target=_drain_output, args=(ollama_process,), daemon=True).start()

# Wait for Ollama to become available
for attempt in range(30):
    try:
        # Check local connection on 127.0.0.1, as Ollama listening on 0.0.0.0 will also listen on 127.0.0.1
        response = requests.get(
            f"{OLLAMA_API}/api/tags",
            timeout=2
        )

        if response.status_code == 200:
            print("Ollama is ready!")
            print(f"API: {OLLAMA_API}")
            break

    except requests.RequestException:
        pass

    time.sleep(1)

else:
    raise RuntimeError("Ollama did not start within 30 seconds.")

Ollama binary: /usr/local/bin/ollama
Ollama is ready!
API: http://127.0.0.1:11434


In [ ]:
response = requests.get(
    f"{OLLAMA_API}/api/tags",
    timeout=10
)

response.raise_for_status()

print("Local Ollama API works!")
print(response.json())

✅ Local Ollama API works!
{'models': [{'name': 'gemma3:4b', 'model': 'gemma3:4b', 'modified_at': '2026-09-03T02:13:41.901123643Z', 'size': 3338801804, 'digest': 'a2af6cc3eb7fa8be8504abaf9b04e88f17a119ec3f04a3addf55f92841195f5a', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'gemma3', 'families': ['gemma3'], 'parameter_size': '4.3B', 'quantization_level': 'Q4_K_M'}, 'capabilities': ['completion']}]}


Pulling the wanted Ollama model(s)

In [ ]:
llm_model = "gemma4:e4b"
subprocess.run(
    [OLLAMA_BIN, "pull", llm_model],
    check=True
)

CompletedProcess(args=['/usr/local/bin/ollama', 'pull', 'gemma3:4b'], returncode=0)

Verify the model

In [ ]:
subprocess.run(
    [OLLAMA_BIN, "list"],
    check=True
)

CompletedProcess(args=['/usr/local/bin/ollama', 'list'], returncode=0)

Setting ngrok authentication token for the API requests.
This is specific to Google Colab. Set the ngrok authentication token in the notebook secrets.

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

# --- 1. Authenticate ---
try:
    NGROK_AUTH = userdata.get("NGROK_AUTH")
except Exception as e:
    raise RuntimeError(
        "Couldn't read the NGROK_AUTH secret. Make sure it exists in the Colab "
        "Secrets panel (key icon in the left sidebar) and that 'Notebook access' "
        "is toggled on for it."
    ) from e

if not NGROK_AUTH:
    raise RuntimeError("NGROK_AUTH was not found in Colab Secrets.")

ngrok.set_auth_token(NGROK_AUTH)

# --- 2. Check if a tunnel for port 11434 already exists ---
existing_tunnel = None
for tunnel in ngrok.get_tunnels():
    if tunnel.public_url and "11434" in tunnel.config.get("addr", ""):
        existing_tunnel = tunnel
        break

if existing_tunnel:
    # Reuse the existing tunnel
    public_url = existing_tunnel.public_url
    print(f"Reusing existing Ollama tunnel: {public_url}")
else:
    # Create a new tunnel
    ollama_tunnel = ngrok.connect(
        addr="127.0.0.1:11434",
        proto="http",
        host_header="localhost:11434",  # Required for Ollama
        # Ollama rejects requests whose Host header isn't localhost/127.0.0.1
        # (DNS-rebinding protection) and returns an empty-body 403. Ngrok would
        # otherwise forward its own public domain as the Host header, so this
        # tells ngrok to rewrite it to the value Ollama expects. This is the
        # exact value Ollama's own docs recommend for ngrok tunnels:
        # https://docs.ollama.com/faq#how-can-i-use-ollama-with-ngrok
    )
    public_url = ollama_tunnel.public_url
    print(f"Created new Ollama tunnel: {public_url}")

Public URL: https://cubicle-bottling-opulently.ngrok-free.dev


Listing all available/pulled models

In [ ]:
!ollama list

NAME         ID              SIZE      MODIFIED       
gemma3:4b    a2af6cc3eb7f    3.3 GB    21 seconds ago    


testing endpoint ngrok

In [ ]:
response = requests.get(
    f"{public_url}/api/tags",
    headers={
        "ngrok-skip-browser-warning": "true"
    },
    timeout=15
)

print("Status:", response.status_code)
print("Response:", response.text)

response.raise_for_status()

print("Public Ollama API is working!")
print(response.json())

Status: 200
Response: {"models":[{"name":"gemma3:4b","model":"gemma3:4b","modified_at":"2026-09-03T02:23:36.678409072Z","size":3338801804,"digest":"a2af6cc3eb7fa8be8504abaf9b04e88f17a119ec3f04a3addf55f92841195f5a","details":{"parent_model":"","format":"gguf","family":"gemma3","families":["gemma3"],"parameter_size":"4.3B","quantization_level":"Q4_K_M"},"capabilities":["completion"]}]}
✅ Public Ollama API is working!
{'models': [{'name': 'gemma3:4b', 'model': 'gemma3:4b', 'modified_at': '2026-09-03T02:23:36.678409072Z', 'size': 3338801804, 'digest': 'a2af6cc3eb7fa8be8504abaf9b04e88f17a119ec3f04a3addf55f92841195f5a', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'gemma3', 'families': ['gemma3'], 'parameter_size': '4.3B', 'quantization_level': 'Q4_K_M'}, 'capabilities': ['completion']}]}


testing model

In [ ]:
response = requests.post(
    f"{public_url}/api/generate",
    headers={
        "Content-Type": "application/json",
        "ngrok-skip-browser-warning": "true"
    },
    json={
        "model": llm_model,
        "prompt": "What is the capital of India?",
        "stream": False
    },
    timeout=120
)

print("Status:", response.status_code)
print("Response:", response.text)

response.raise_for_status()

print("\nModel response:")
print(response.json()["response"])

Status: 200
Response: {"model":"gemma3:4b","created_at":"2026-09-03T02:36:50.327809327Z","response":"The capital of India is **New Delhi**. \n\nIt's a separate city created specifically to be the capital, and it's divided into two parts:\n\n*   **New Delhi:** The main, historic part containing many government buildings and iconic landmarks.\n*   **Delhi:** The larger metropolitan area that surrounds New Delhi.","done":true,"done_reason":"stop","context":[105,2430,107,3689,563,506,5279,529,4673,236881,106,236743,107,105,2028,107,818,5279,529,4673,563,5213,4199,18390,84750,236743,108,1509,236789,236751,496,7732,3207,4464,10916,531,577,506,5279,236764,532,625,236789,236751,11310,1131,1156,4688,236787,108,236829,139,1018,4199,18390,53121,669,1689,236764,17110,912,7906,1551,3251,10692,532,22799,70836,236761,107,236829,139,1018,89726,53121,669,6268,49880,2433,600,79615,1799,18390,236761],"total_duration":109828846100,"load_duration":69670987871,"prompt_eval_count":17,"prompt_eval_duration":3

Example call to an Ollama model using curl

In [ ]:
# NOTE: we export public_url as a shell env var and use a %%bash cell instead of
# "!curl ... {public_url}". With "!", IPython treats EVERY {...} on the line as a
# Python variable to interpolate -- including the curly braces inside the JSON
# body itself (e.g. {"model": ...}) -- which breaks the command. %%bash cells
# don't do that interpolation, so the JSON is passed through untouched.
os.environ["OLLAMA_PUBLIC_URL"] = public_url

In [ ]:
!curl -X POST "https://cubicle-bottling-opulently.ngrok-free.dev/api/generate" -H "Content-Type: application/json" -d '{"model": "gemma3:4b", "prompt": "What is the capital of India?", "stream": false}'



Example call to an Ollama model using requests

In [ ]:
import requests
import json

# Define the URL and headers
url = f"{public_url}/api/generate"
headers = {
    "Content-Type": "application/json"
}

# Define the data (payload) to send in the POST request
data = {
    "model": llm_model,
    "prompt": "Who is Sundar Pichai?",
    # stream=False so the response body is a single JSON object that
    # response.json() can parse. With stream=True, Ollama returns a series of
    # newline-delimited JSON objects, and response.json() raises a
    # JSONDecodeError because that's not valid single-object JSON.
    "stream": False
}

# Send the POST request
response = requests.post(url, headers=headers, data=json.dumps(data))

# Check if the request was successful and print the response
if response.status_code == 200:
    print("Response:", response.json()["response"])
else:
    print("Failed to get a response. Status code:", response.status_code)
    print(response.text)

Response: Sundar Pichai is a hugely influential figure in the tech world, currently serving as the **CEO of Google and Alphabet Inc.** Let's break down who he is and what he does:

**Key Facts:**

* **Born:** June 10, 1972, in Chennai, India.
* **Education:** He earned a Bachelor's degree in Metallurgical Engineering from the Indian Institute of Technology Kharagpur, followed by a Master of Engineering from Stanford University.
* **Early Career at Google:** He joined Google in 2004 and quickly rose through the ranks, initially working on Google Talk (now Google Chat) and Google Docs.
* **Key Leadership Roles:** He’s held several crucial positions within Google, including:
    * **Vice President of Products and User Experiences:** He oversaw a large portfolio of Google products, including Search, Maps, YouTube, and Google Voice.
    * **Google's Chief Operating Officer (COO):** This was a pivotal role where he was responsible for Google’s overall operations, strategy, and execution.
   